# Harvest Slot DL 신선도 판별 체크리스트

이 노트북은 DL 담당자가 단계별로 진행 상황을 확인하면서 작업하기 위한 문서입니다.

현재 MVP 방향은 **확장을 염두에 두되, 1차 구현은 사과(apple) 신선도 판별**로 고정합니다.

DL 결과는 출고를 자동 결정하는 값이 아니라, **점주가 최종 선별/출고 판단을 하기 위한 보조 지표**입니다.

## 0. 전체 진행 체크리스트

- [ ] 1단계: DL 범위 확정
- [ ] 2단계: 폴더 구조 생성
- [ ] 3단계: 데이터 수집 및 라벨 기준 확정
- [ ] 4단계: 이미지 전처리
- [ ] 5단계: OpenCV 보조 특징 추출 구현
- [ ] 6단계: PyTorch ResNet18 전이학습 구성
- [ ] 7단계: 단일 이미지 추론 함수 구현
- [ ] 8단계: FastAPI 업로드 API 연동
- [ ] 9단계: DB 저장 구조 연동
- [ ] 10단계: 점주 앱 카메라/갤러리 업로드 연동
- [ ] 11단계: 테스트 이미지 세트와 발표 데모 준비

## 1단계. DL 범위 확정

첫 번째로 할 일은 구현 범위를 고정하는 것입니다.

### 확정안

| 항목 | 내용 |
|---|---|
| 1차 대상 과일 | 사과 |
| fruit_type | `apple` |
| 대표 품종 | 후지, `fuji` |
| 라벨 | `A`, `B`, `C` |
| QC 등급 매핑 | A=특, B=상, C=보통 |
| 데이터 split | 객체 단위 train 70 / valid 10 / test 20 |
| 기본 학습 설정 | ResNet18, batch 64, epoch 15, patience 5 |
| 출력 판단 | `PASS`, `REVIEW`, `HOLD` |
| 모델 역할 | 품질/신선도 판별 보조 |
| 최종 결정 | 점주가 확정 |

### 기능 정의 문장

> 사과 이미지를 입력받아 PyTorch 기반 모델과 OpenCV 보조 특징을 사용해 품질 등급 A/B/C, 신선도 점수, 멍 확률, 출고 보조 판단 PASS/REVIEW/HOLD를 반환한다. 단, 결과는 자동 출고 결정이 아니라 점주 최종 판단을 돕는 보조 지표로 사용한다.

이 단계가 확정되면 다음 단계는 폴더 구조 생성입니다.

In [ ]:
# 1단계 설정값
TARGET_FRUIT = "apple"
TARGET_VARIETY = "fuji"
LABELS = ["A", "B", "C"]
DECISIONS = ["PASS", "REVIEW", "HOLD"]

print("대상 과일:", TARGET_FRUIT)
print("대표 품종:", TARGET_VARIETY)
print("라벨:", LABELS)
print("출고 보조 판단:", DECISIONS)

## 2단계. 폴더 구조 생성

확장을 염두에 두기 위해 `train/apple/A`처럼 과일 종류를 한 단계 포함합니다.

권장 구조:

```text
ai/dl/freshness/
  dataset.py
  train.py
  infer.py
  opencv_features.py
  models/
  data/
    raw/
    processed/
    train/apple/A
    train/apple/B
    train/apple/C
    valid/apple/A
    valid/apple/B
    valid/apple/C
    test/apple/A
    test/apple/B
    test/apple/C
```

나중에 배나 감귤을 추가하면 `train/pear/A`, `train/citrus/A`처럼 늘릴 수 있습니다.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Note" else Path.cwd()
DL_ROOT = PROJECT_ROOT / "ai" / "dl" / "freshness"

paths = [
    DL_ROOT / "models",
    DL_ROOT / "data" / "raw",
    DL_ROOT / "data" / "processed",
]

for split in ["train", "valid", "test"]:
    for label in LABELS:
        paths.append(DL_ROOT / "data" / split / TARGET_FRUIT / label)

for path in paths:
    path.mkdir(parents=True, exist_ok=True)

print("생성 완료:", DL_ROOT)

## 3단계. 데이터 수집 및 라벨 기준

MVP 기준:

- 최소 100장, 목표 300장
- 사과 우선
- 배경은 단색 권장
- 조명은 최대한 일정하게
- 각도는 정면/측면/상단 섞어서 촬영

라벨 기준 초안:

| 라벨 | 의미 |
|---|---|
| A | QC 등급 `특`, 가장 좋은 품질 등급 |
| B | QC 등급 `상`, 중간 품질 등급 |
| C | QC 등급 `보통`, 기본 품질 등급 |

중요: 이 기준은 QC 품질 등급을 프로젝트의 신선도 판별 대리 라벨로 사용하는 것입니다.

In [ ]:
# 현재 데이터 개수 확인
for split in ["train", "valid", "test"]:
    print(f"[{split}]")
    for label in LABELS:
        folder = DL_ROOT / "data" / split / TARGET_FRUIT / label
        image_count = sum(1 for p in folder.glob("*.*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
        print(f"  {label}: {image_count} images")

## 4단계 이후 구현 메모

이 아래 단계는 실제 데이터가 들어온 뒤 진행합니다.

- 이미지 전처리: 손상 이미지 제거, 224x224 resize
- OpenCV 특징: 색 선명도, 둥근 정도, 멍 의심 확률
- PyTorch 학습: ResNet18 전이학습
- 기본 설정: batch size 64, epoch 15, patience 5
- Kaggle T4에서 메모리가 충분하면 batch size 128 시도
- 추론 결과: CNN 등급 + OpenCV 점수 + rule 기반 최종 판단
- API 연동: `/api/v1/dl/freshness-scan`
- 데모 준비: 테스트 이미지 고정, 5초 이내 응답 확인

## 점수 산정 초안

```text
freshness_score =
  CNN 등급 점수 * 0.50
+ 색 선명도 * 0.25
+ 둥근 정도 * 0.15
+ 멍 없음 점수 * 0.10
```

출고 보조 판단:

- `PASS`: score >= 80 and bruise_probability < 0.2
- `REVIEW`: score >= 60
- `HOLD`: score < 60 or bruise_probability >= 0.5

In [ ]:
def make_shipping_decision(freshness_score: float, bruise_probability: float) -> str:
    if freshness_score < 60 or bruise_probability >= 0.5:
        return "HOLD"
    if freshness_score >= 80 and bruise_probability < 0.2:
        return "PASS"
    return "REVIEW"

for score, bruise in [(91.3, 0.07), (72.0, 0.18), (58.0, 0.12), (82.0, 0.55)]:
    print(score, bruise, "=>", make_shipping_decision(score, bruise))